# 2장 실습 — 도로망을 그래프로 만들기

하남시 도로망을 읽어 자동차용 인접 리스트를 만들고, 하남시청과 미사역의 출발·도착 노드를 구합니다.
교재 2장에서 살펴본 연결 관계, 통행 방향, 통행시간을 코드로 확인합니다.

위에서부터 순서대로 실행합니다. DTUMOS 서버는 필요하지 않습니다.
6절의 `None`은 직접 채우는 자리이며, 비워 두어도 끝까지 실행됩니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from IPython.display import display
from lab import expect, todo

## 1. 노드와 엣지 표 읽기 (교재 2.2)

`read_parquet`으로 두 파일을 읽습니다. 먼저 노드 식별자와 좌표를 확인합니다.

In [ ]:
import pandas as pd
from smartmob.data import data_path

nodes = pd.read_parquet(data_path("hanam/road_graph_nodes.parquet"))
edges = pd.read_parquet(data_path("hanam/road_graph_edges.parquet"))

print(f"원본: 노드 {len(nodes):,}개, 엣지 {len(edges):,}개")
nodes[["node_id", "lat", "lon"]].head(3)

`node_id`가 노드 식별자이며, `lat`은 위도, `lon`은 경도입니다.
원본에는 노드 21,931개와 엣지 59,873개가 있습니다.
다음은 엣지의 길이(m), 자유류 속도(km/h), 일방통행 여부를 확인합니다.

In [ ]:
edges[["edge_id", "highway", "length", "free_flow_speed_kmh", "oneway"]].head(3)

첫 행의 `motorway_link`는 고속도로 연결로입니다. 이어지는 두 행은 같은 주택가 구간의 서로 반대 방향입니다.
각 행은 한 방향의 이동을 나타냅니다.

## 2. 출발·도착 노드 확인하기 (교재 2.2~2.3)

이 파일은 양 끝 노드를 별도 컬럼 대신 `edge_id`에 넣었습니다.
예를 들어 `e37375263_f_445273230_436257996`의 마지막 두 숫자가 출발·도착 노드 번호입니다.
각 번호 앞에 `n`을 붙이면 노드 표의 식별자가 됩니다. 이는 이 실습 파일의 저장 규칙입니다.

`rsplit("_", 2)`로 뒤의 두 번호를 꺼내 `source`, `target` 컬럼을 만듭니다.

In [ ]:
def parse_edge_id(edge_id):
    _, source_osm, target_osm = edge_id.rsplit("_", 2)
    return f"n{source_osm}", f"n{target_osm}"


endpoints = edges["edge_id"].map(parse_edge_id)
edges[["source", "target"]] = pd.DataFrame(endpoints.tolist(), index=edges.index)
edges[["edge_id", "source", "target", "direction", "oneway"]].head(3)

두 번째와 세 번째 행은 `source`와 `target`이 서로 뒤바뀌어 있습니다.
양방향 구간의 두 방향이 이미 저장되어 있으므로 인접 리스트를 만들 때 역방향을 추가하지 않습니다.

## 3. 자동차용 엣지 선택하기 (교재 2.3)

`DRIVE_HIGHWAYS`는 학습용 그래프에서 자동차용으로 선택하는 도로 종류입니다.
같은 필터를 적용한 뒤 어떤 종류가 남고 제외되는지 셉니다.

In [ ]:
from smartmob.teaching.graph import DRIVE_HIGHWAYS

is_drive = edges["highway"].isin(DRIVE_HIGHWAYS)
drive_edges = edges.loc[is_drive].copy().reset_index(drop=True)

by_type = edges.groupby("highway").size().sort_values(ascending=False).to_frame("엣지 수")
by_type["자동차용으로 선택"] = by_type.index.isin(DRIVE_HIGHWAYS)
display(by_type)
print(f"선택 {len(drive_edges):,}개 / 제외 {len(edges) - len(drive_edges):,}개")

자동차용 엣지 28,589개가 남습니다. `footway`, `cycleway`, `steps` 등은 제외됩니다.
이 필터는 도로 종류만 사용하므로 개별 도로의 출입 제한이나 회전 제한까지 확인한 결과는 아닙니다.

그래프를 만들기 전에 선택한 엣지의 양 끝 노드가 노드 표에 있는지 확인합니다.

In [ ]:
coord = {r.node_id: (r.lat, r.lon) for r in nodes.itertuples(index=False)}
missing_endpoint = (
    ~drive_edges["source"].isin(coord) | ~drive_edges["target"].isin(coord)
)
print(f"양 끝 노드 중 하나라도 없는 엣지: {missing_endpoint.sum():,}개")
assert not missing_endpoint.any(), "노드 표에 없는 출발·도착 노드를 확인합니다."

이 데이터에서는 0개입니다. 선택한 모든 엣지의 양 끝 좌표를 찾을 수 있습니다.

## 4. 통행시간과 인접 리스트 만들기 (교재 2.4)

통행시간을 계산하기 전에 길이와 속도가 유한한 양수인지 확인합니다.
`isna()`는 결측값을, `<= 0`은 0 이하의 값을 찾습니다.

In [ ]:
import numpy as np

length = drive_edges["length"]
speed = drive_edges["free_flow_speed_kmh"]
invalid_length = length.isna() | ~np.isfinite(length) | (length <= 0)
invalid_speed = speed.isna() | ~np.isfinite(speed) | (speed <= 0)

print(f"길이 확인 대상: {invalid_length.sum():,}개")
print(f"속도 확인 대상: {invalid_speed.sum():,}개")
assert not (invalid_length | invalid_speed).any(), "길이·속도 값을 확인한 뒤 계산합니다."

두 값 모두 0개이므로 주어진 길이와 속도를 그대로 씁니다.
`3.6 × 길이(m) / 속도(km/h)`로 통행시간(초)을 계산합니다.

In [ ]:
drive_edges["travel_time_s"] = 3.6 * length / speed
drive_edges[["source", "target", "length", "free_flow_speed_kmh", "travel_time_s"]].head(3)

첫 엣지는 약 727m를 60km/h로 이동하므로 약 43.6초입니다.
이제 각 출발 노드에 `(도착 노드, 통행시간 초, 엣지 인덱스)`를 모읍니다.

In [ ]:
adj = {}
for edge_index, row in drive_edges.iterrows():
    adj.setdefault(row["source"], []).append(
        (row["target"], row["travel_time_s"], edge_index)
    )
    # 들어오는 엣지만 있는 노드도 그래프에 포함합니다.
    adj.setdefault(row["target"], [])

expect("자동차 도로 노드", len(adj), 12_566)
expect("자동차 도로 엣지", sum(len(neighbors) for neighbors in adj.values()), 28_589)

노드 12,566개와 엣지 28,589개가 만들어집니다.
`edge_index`는 필터 후 인덱스를 다시 매긴 `drive_edges`의 행 위치입니다.
원본 파일의 행 번호와는 다릅니다.

한 노드에서 나가는 엣지를 골라 원래 길이·속도와 함께 확인합니다.

In [ ]:
sample = next(node for node, neighbors in adj.items() if len(neighbors) >= 3)
print(f"출발 노드: {sample}")
for neighbor, seconds, edge_index in adj[sample]:
    edge = drive_edges.iloc[edge_index]
    print(
        f"  → {neighbor}: {seconds:.1f}초 "
        f"({edge['length']:.0f}m, {edge['free_flow_speed_kmh']:.0f}km/h)"
    )

각 줄이 이 노드에서 바로 이동할 수 있는 이웃과 그 구간의 통행시간입니다.
3장에서는 이런 목록을 이어서 읽으며 목적지까지의 경로를 찾습니다.

`load_road_graph`도 같은 필터와 통행시간 계산으로 인접 리스트를 만듭니다.
직접 만든 그래프와 크기를 비교하고, 같은 노드의 이웃 목록을 읽어 봅니다.

In [ ]:
from smartmob.data import load_road_graph

drive = load_road_graph("hanam", modes=("drive",))
expect("노드 수 일치", drive.n_nodes, len(adj))
expect("엣지 수 일치", drive.n_edges, len(drive_edges))
pd.DataFrame(drive.neighbors(sample), columns=["이웃 노드", "통행시간(초)", "엣지 인덱스"])

위에서 직접 만든 목록과 같은 이웃·통행시간이 나옵니다.
3장부터는 `load_road_graph`로 읽고, `drive.neighbors(노드)`로 이웃 목록을 사용합니다.

## 5. 좌표를 자동차 도로망에 연결하기 (교재 2.5)

`nearest_node`에 `(위도, 경도)` 순서로 좌표를 전달합니다.
선택한 노드와 입력 좌표 사이의 거리도 확인합니다.

In [ ]:
from smartmob.teaching.graph import haversine_km

HANAM_CITY_HALL = (37.5393, 127.2148)
MISA_STATION = (37.5606, 127.1930)

start = drive.nearest_node(*HANAM_CITY_HALL)
goal = drive.nearest_node(*MISA_STATION)

snaps = []
for name, point, node in [
    ("하남시청", HANAM_CITY_HALL, start),
    ("미사역", MISA_STATION, goal),
]:
    distance_m = haversine_km(*point, *drive.coord[node]) * 1000
    snaps.append({"지점": name, "선택한 노드": node, "좌표와 노드 사이(m)": distance_m})

pd.DataFrame(snaps)

두 좌표 모두 선택한 노드까지 약 43m 떨어져 있습니다.
`haversine_km`은 두 좌표 사이의 지표면 거리를 km로 계산하며, 도로를 따라가는 거리는 아닙니다.

`nearest_node`는 자동차 그래프에 포함된 노드만 후보로 씁니다.
SciPy가 있으면 노드 좌표로 KD-트리를 한 번 만들어 재사용하고, 없으면 모든 후보를 순회합니다.
두 방식 모두 위경도 차이를 사용하므로 지표면 거리상 가장 가까운 노드를 보장하지는 않습니다.
노드 사이의 경로에는 위 표의 간격을 이동하는 시간이 포함되지 않습니다.

## 6. 직접 계산하기

### 6.1 속도가 바뀌면 경로 비교가 어떻게 달라지는가

교재 2.4절의 가상 도로망을 계산합니다.
A → B → C는 200m 구간 두 개를 30km/h로, A → D → C는 350m 구간 두 개를 60km/h로 이동합니다.
D를 거치는 두 구간의 속도가 모두 30km/h가 된 경우도 계산합니다.

In [ ]:
via_b_s = None          # B를 거치는 경로의 통행시간(초)
via_d_s = None          # D를 거치는 경로의 통행시간(초)
via_d_slow_s = None     # D를 거치는 두 구간이 모두 30km/h일 때의 통행시간(초)

expect("B 경유", via_b_s, 48.0, tol=1e-9)
expect("D 경유", via_d_s, 42.0, tol=1e-9)
expect("D 경유, 속도 감소 후", via_d_slow_s, 84.0, tol=1e-9)

속도 변경 전후에 어느 경로가 더 빠른지 한 문장으로 적어 봅시다.

### 6.2 일방통행 엣지 비율

`drive_edges["oneway"]`가 참인 행의 비율을 구합니다.
참·거짓 컬럼의 `mean()`을 이용할 수 있습니다. 이 비율은 실제 도로 개수가 아닌 방향별 엣지 수를 기준으로 합니다.

In [ ]:
oneway_share = None     # 일방통행 엣지 수 / 자동차용 전체 엣지 수 (0~1)

expect("일방통행 엣지 비율", oneway_share, 2_413 / 28_589, tol=1e-9)

모든 엣지에 역방향을 추가하면 어떤 잘못된 이동이 가능해지는지 한 문장으로 적어 봅시다.

### 6.3 다른 호출 좌표 연결하기

하남시청 좌표를 기준으로 위도에 0.001을 더한 가상 호출 지점을 사용합니다.
`drive.nearest_node`로 노드를 찾고, `haversine_km`으로 호출 좌표와 선택한 노드 사이의 거리를 구합니다.

In [ ]:
call_point = (HANAM_CITY_HALL[0] + 0.001, HANAM_CITY_HALL[1])
call_node = None          # 자동차 그래프에서 선택한 노드
snap_distance_m = None    # 호출 좌표에서 선택한 노드까지의 지표면 거리(m)

todo("호출 지점의 노드", call_node)
todo("좌표와 노드 사이 거리(m)", snap_distance_m)
if call_node is not None:
    expect("자동차 그래프에 포함", call_node in drive.nodes, True)
    expect("가까운 노드 선택", call_node, drive.nearest_node(*call_point))
    if call_node in drive.nodes:
        expected_m = haversine_km(*call_point, *drive.coord[call_node]) * 1000
        expect("거리(m)", snap_distance_m, expected_m, tol=0.1)

계산한 거리가 크다면 노드 사이의 경로만으로 도착시간을 안내할 때 무엇이 빠지는지 적어 봅시다.

## 3장으로 가져갈 것

- `drive`는 자동차용 방향 그래프이며, `drive.neighbors(노드)`는 이웃·통행시간(초)·엣지 인덱스를 돌려줍니다.
- `start`, `goal`은 하남시청과 미사역 좌표에 대응하는 노드입니다.
- 3장에서는 이 두 노드 사이에서 엣지 통행시간의 합이 가장 작은 경로를 구합니다.